# C7-cnn-transfer — Practice p18 — Solution

The lesson's normalized roughness averages horizontal and vertical neighbor differences, then divides by twice the per-map mean absolute deviation.

In [ ]:
import numpy as np

SEED = 20260804
rng = np.random.default_rng(SEED)

def _box5(m):
    out = np.zeros((m.shape[0] - 4, m.shape[1] - 4))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            out[i, j] = m[i:i + 5, j:j + 5].mean()
    return out

noise = rng.standard_normal((6, 30, 30))
stack_A = np.maximum(np.stack([_box5(_box5(ch)) for ch in noise]) - 0.06, 0.0)
stack_B = np.maximum(rng.standard_normal((6, 30, 30)), 0.0)

def act_frac(stack):
    return float((stack > 0).mean())

def rough(stack):
    dx = np.abs(np.diff(stack, axis=-1)).mean()
    dy = np.abs(np.diff(stack, axis=-2)).mean()
    scale = np.abs(stack - stack.mean(axis=(-2, -1), keepdims=True)).mean()
    return float((dx + dy) / (2 * scale))

af_A = act_frac(stack_A)
af_B = act_frac(stack_B)
rough_A = rough(stack_A)
rough_B = rough(stack_B)
label_A = "late"
label_B = "early"
deeper_shape = (1, 2048, 7, 7)


Stack A has the lower activation fraction and lower normalized roughness, so it
matches sparse, slowly varying late semantic maps; stack B matches denser,
high-frequency early maps. Receptive fields grow with depth while spatial grids
shrink and channel counts grow, so `(1, 2048, 7, 7)` is the deeper shape.


### Answer check

In [ ]:
assert 0.0 < af_A < af_B < 1.0
assert 0.0 < rough_A < rough_B
assert (label_A, label_B) == ("late", "early")
assert deeper_shape == (1, 2048, 7, 7)
assert np.isclose(af_A, 0.38360881542699726)
assert np.isclose(af_B, 0.4988888888888889)
assert np.isclose(rough_A, 0.2950881566137611)
assert np.isclose(rough_B, 1.2367143595187606)
